# Phase 0.7 — π0.5 on Colab (L4)

A **second, smaller π0.5 run** on a Colab L4, to sit alongside the full run on the rented 4090
([colab_phase07_pi05.ipynb](colab_phase07_pi05.ipynb)). Everything from the runtime check onward is
copied verbatim from that notebook — same dataset, same holdout `[9, 19, 29, 39, 49]`, same seed, same
`train_expert_only` + bf16 + gradient checkpointing — so the two are directly comparable at matched
steps.

## Does an L4 fit? Yes on memory, and that is the easy part

π0.5 peaked at **18.18 GiB** with gradient checkpointing on, so a 24 GB L4 has ~6 GiB of headroom.
**Keep `gradient_checkpointing=true`** — that figure is *with* it, and turning it off to gain speed would
likely exceed the card.

Two things are **not** fine on Colab, and this notebook differs from the 4090 one because of them:

| | 4090 host | **Colab L4** |
|---|---|---|
| Steps | 30,000 (18.0 epochs) | **10,000 (6.0 epochs)** |
| Projected wall clock | ~17–25 h | **~14–19 h** |
| Checkpoints kept locally | all 6 | **newest 2** (reaped; all are still pushed to the Hub) |
| `save_freq` | 5,000 | **2,500** |

**Why fewer steps.** The L4 is roughly 2.5–3.5× slower than the A100 that measured 1.95 s/step, so 30,000
steps would be **~41–57 h** — past any Colab session, and a lot of reconnects even though Part C resumes.
10,000 steps lands around 14–19 h. Part B measures the real rate in ~20 min and prints the projection;
raise `BASE_STEPS` if your L4 turns out faster than this estimate.

**Why reaping is back on.** Measured on these runtimes, Colab has 112.6 GB of disk with ~44 GB taken by
the base image, leaving **~68 GB**. Six π0.5 checkpoints at 11.55 GB each plus the 14.5 GB base model is
**84.5 GB** — it does not fit, and the π0.5 smoke test alone has already driven a Colab disk to 94 GB.
Keeping the newest two is ~38 GB and leaves 30 GB spare. Nothing is lost: every checkpoint is still
pushed to the Hub before the local copy is removed.

**Is 6 epochs enough?** Unknown for π0.5, but not obviously too few. SmolVLA's held-out loss on this same
50-episode dataset bottomed at **12 epochs and rose 61% by 48** — if π0.5 overfits on a similar schedule,
6 epochs may be near its optimum rather than short of it. The 2,500-step checkpoints bracket it either
way.

> Pushes to **`pi05_tray_cylinder_s1000_trim_L4`**, a different Hub repo from the 4090 run, so the two
> cannot collide or resume from each other.

---
# Part A — Setup

## 1 — Runtime

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", GPU)
assert GPU, "No GPU visible. On Colab: Runtime > Change runtime type > GPU (A100)."
if "T4" in GPU:
    print("\n!! T4 (16 GB): too small. pi0.5 peaks at 18.2 GiB. Use an L4.")
elif "L4" in GPU:
    print("\nL4 (24 GB): fits - pi0.5 peaked at 18.2 GiB on an A100 with gradient checkpointing on,")
    print("            leaving ~6 GiB of headroom here. Speed is the constraint, not memory:")
    print("            see the step count note above.")
elif "A100" in GPU:
    print("\nA100: you got the better card. Raise STEPS back toward 30000 if you want the full")
    print("      18-epoch recipe - at ~1.95 s/step that is ~16 h rather than the ~46 h an L4 needs.")


## 2 — Install

Colab already provides a working CUDA torch, so only lerobot is installed — pinned to **v0.6.1**, the
version that recorded the dataset. (The 4090 notebook installs torch itself because that host is bare;
here, letting Colab's torch stand is both simpler and less likely to break.)

In [ ]:
!pip install -q "lerobot[pi,dataset,training]==0.6.1"
import lerobot, torch
print("lerobot", lerobot.__version__, "| torch", torch.__version__, "| cuda", torch.version.cuda)


> If `import lerobot` fails, restart the runtime (**Runtime > Restart session**) and re-run this cell.
> Nothing later imports lerobot into this kernel — every use goes through a subprocess.

## 5 — Authentication

**A Colab secret alone is not enough.** `huggingface_hub` resolves a token as OIDC → `HF_TOKEN` env var →
token file → Colab secret, and that last path calls `google.colab.userdata.get()`, which needs the
kernel's channel to the Colab frontend. `lerobot-train` is a **subprocess**, where that channel does not
exist — so with only a secret set, `whoami()` succeeds in the notebook while training authenticates as
nobody. These cells copy the token into the environment *and* the token file, then prove a subprocess can
use it.

The run also creates a new model repo. A fine-grained token can be granted `repo.write` either globally
or **scoped to your own user** — the latter is the least-privilege setup and shows up nowhere under
`global`, so the cell checks both and only *warns* if it finds neither. Part B settles it for real by
pushing a checkpoint and verifying it landed.

In [ ]:
import os, sys, subprocess
from huggingface_hub import get_token, login, whoami

tok = os.environ.get("HF_TOKEN") or get_token()
if not tok:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None
if not tok:
    from huggingface_hub import notebook_login
    notebook_login(); tok = get_token()
assert tok, "no HF token - add a Colab secret named HF_TOKEN, or run notebook_login()"

login(token=tok, add_to_git_credential=False)     # writes ~/.cache/huggingface/token
os.environ["HF_TOKEN"] = tok                      # set AFTER login, so it does not warn that the
                                                  # env var overrides what was just configured
                                                  # (same token either way); subprocesses inherit it

r = subprocess.run([sys.executable, "-c",
                    "from huggingface_hub import whoami; print(whoami()['name'])"],
                   capture_output=True, text=True)
assert r.returncode == 0, "a subprocess still cannot authenticate:\n" + r.stderr[-800:]
me = whoami()
print("HF ok - notebook:", me["name"], "| subprocess:", r.stdout.strip())


def hf_write_grant(me):
    """Report how (or whether) this token is allowed to create repos in the user's namespace.

    A fine-grained token carries permissions either globally or SCOPED TO AN ENTITY. The
    least-privilege setup - and the one the Hub's UI steers you to - is `repo.write` scoped to your
    own user, which does not appear under `global` at all. Checking only `global` therefore rejects
    a perfectly good token, which is exactly what an earlier version of this cell did.
    """
    auth = me.get("auth", {}).get("accessToken", {})
    if auth.get("role") != "fineGrained":
        return True, f"{auth.get('role', 'write')} token"
    fg = auth.get("fineGrained") or {}
    if "repo.write" in (fg.get("global") or []):
        return True, "global repo.write"
    for s in fg.get("scoped") or []:
        ent = s.get("entity") or {}
        if (ent.get("type") in ("user", "org") and ent.get("name") == me["name"]
                and "repo.write" in (s.get("permissions") or [])):
            return True, f"repo.write scoped to {ent['type']} '{ent['name']}'"
    return False, f"no repo.write found - global={fg.get('global')}"


ok, how = hf_write_grant(me)
print("token write access:", how)
if not ok:
    # Deliberately a warning, not an assertion. The Hub's permission model has more shapes than it
    # is wise to enumerate here, and Part B proves the real thing 15 minutes from now by pushing a
    # checkpoint and checking it landed. A hard gate here can only produce false blocks.
    print("\n!! Could not confirm this token can create the model repo this run pushes to.")
    print("   If Part B's 'checkpoint uploaded to the Hub' check fails, that is the reason - give")
    print("   the token repo.write (globally, or scoped to your user) at")
    print("   https://huggingface.co/settings/tokens")

In [ ]:
import os, sys, subprocess, wandb

key = os.environ.get("WANDB_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        key = None
if key:
    os.environ["WANDB_API_KEY"] = key; wandb.login(key=key)
else:
    wandb.login()                                  # key from https://wandb.ai/authorize
    os.environ["WANDB_API_KEY"] = wandb.api.api_key

r = subprocess.run([sys.executable, "-c", "import wandb; print(bool(wandb.api.api_key))"],
                   capture_output=True, text=True)
assert r.stdout.strip() == "True", "a subprocess cannot see the wandb key:\n" + r.stderr[-500:]
print("wandb ok in notebook and subprocess")

## 6 — Configuration


| Knob | Value | Why |
|---|---|---|
| Base | `lerobot/pi05_base` | The published checkpoint |
| Batch / steps | 16 / 30,000 | 480k samples ≈ 16 epochs — the same sample budget as the ACT run |
| Precision | bf16 + gradient checkpointing | Required to fit 40 GB; the LeRobot π0.5 recipe uses both |
| Adaptation | `train_expert_only=true`, vision encoder frozen | Matches SmolVLA's regime (see above). Full fine-tune needs >70 GB |
| LR / schedule | 2.5e-5, cosine with 1k warmup | Policy preset. `scheduler_decay_steps` pinned to the real step count |
| Images | 224×224, both cameras + one padded empty | Base config; the unfed third camera gets a zero mask |

In [ ]:
import collections, json, os, re, shutil, subprocess, sys, threading, time
from pathlib import Path

# Colab has one big disk and no separate data volume, so no redirection is needed - this is where
# the work-dir cell of the 4090 notebook would otherwise have put these.
WORK = Path("/content")
OUTPUT_ROOT = WORK / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_USER = "HALDijkstraaa"
DATASET = f"{HF_USER}/so101_tray_cylinder_portable_20261003_211912_trimmed"
PROJECT = "phase07-policy-transfer"
SEED    = 1000

# Held out for validation loss. Episodes 0-44 are 9 tray-spot blocks of 5 CONSECUTIVE episodes and
# 45-49 are off-grid, so lerobot's default "last ceil(n*eval_split) episodes" split would hold out
# exactly the 5 off-grid episodes - removing the only unmarked placements from training. The split
# slices the tail of --dataset.episodes IN THE ORDER GIVEN, so putting the holdout last reorders it
# into a balanced one: the last episode of blocks 2, 4, 6, 8 plus one off-grid.
HOLDOUT  = [9, 19, 29, 39, 49]
EPISODES = [e for e in range(50) if e not in HOLDOUT] + HOLDOUT

EVAL_SPLIT, LOG_FREQ = 0.1, 200

# Keep every checkpoint on local disk. They are each the whole model plus optimizer state, so this
# is the setting that decides whether the run fits the disk - Part B projects the total before you
# start, and a watchdog re-checks it during the run and shouts before it runs out. Flip to False to
# retain only the newest KEEP_CKPTS instead; nothing is lost either way because every checkpoint is
# also pushed to the Hub.
KEEP_ALL_CHECKPOINTS = False
KEEP_CKPTS = 2

# ---- pi0.5-specific ----------------------------------------------------------------------------
JOB   = "pi05_tray_cylinder_s1000_trim_L4"
# The published recipe. These two never change; BATCH/STEPS below are what the run actually uses, and
# Part B derives them from these so re-running it is idempotent rather than compounding.
BASE_BATCH, BASE_STEPS = 16, 10000
BATCH, STEPS = BASE_BATCH, BASE_STEPS
SAVE_FREQ = 2500      # how often a checkpoint is written AND pushed
EVAL_STEPS = 2500     # held-out loss cadence; training is unaffected either way.
# SAVE_FREQ must be a multiple of EVAL_STEPS, so that every checkpoint has an eval at its own
# step: lerobot tests `step % eval_steps == 0` and `step % save_freq == 0` independently, and
# runs the eval first, so a mismatched pair leaves some checkpoints with no loss beside them.
assert SAVE_FREQ % EVAL_STEPS == 0, f"{SAVE_FREQ} is not a multiple of {EVAL_STEPS}"
BASE_MODEL = "lerobot/pi05_base"

# The pretrained checkpoint declares its own camera names; the batch is renamed to match.
RENAME = {
    "observation.images.top": "observation.images.base_0_rgb",
    "observation.images.wrist": "observation.images.left_wrist_0_rgb"
}

NUM_WORKERS = min(8, os.cpu_count() or 2)


def train_cmd(job=None, steps=None, batch=None, extra=()):
    """Build the lerobot-train argv.

    Every default is resolved from the globals at CALL time, not bound at definition time, so a
    batch step-down in Part B reaches a later bare train_cmd() call. Checkpoints go to the Hub every
    SAVE_FREQ steps, so a disconnect costs at most that many.
    """
    job = JOB if job is None else job
    steps = STEPS if steps is None else steps
    batch = BATCH if batch is None else batch
    return ["lerobot-train",
        f"--dataset.repo_id={DATASET}", f"--dataset.episodes={EPISODES}",
        f"--dataset.eval_split={EVAL_SPLIT}", f"--eval_steps={EVAL_STEPS}",
        *( [f"--dataset.video_backend={globals()['VIDEO_BACKEND']}"]
           if globals().get("VIDEO_BACKEND") else [] ),
        f"--policy.path={BASE_MODEL}",
        f"--rename_map={json.dumps(RENAME)}",
        "--policy.dtype=bfloat16",
        "--policy.gradient_checkpointing=true",
        "--policy.train_expert_only=true",
        "--policy.freeze_vision_encoder=true",

        "--policy.device=cuda",
        "--policy.push_to_hub=true", f"--policy.repo_id={HF_USER}/{job}", "--policy.private=true",
        "--save_checkpoint_to_hub=true",
        # Pin the cosine decay to the real step count so the schedule always completes,
        # whatever batch the smoke test settles on (the preset default is 30k).
        f"--policy.scheduler_decay_steps={steps}",
        f"--batch_size={batch}", f"--steps={steps}", f"--num_workers={NUM_WORKERS}",
        f"--seed={SEED}", f"--save_freq={SAVE_FREQ}", f"--log_freq={LOG_FREQ}",
        f"--output_dir={OUTPUT_ROOT}/{job}", f"--job_name={job}",
        "--wandb.enable=true", f"--wandb.project={PROJECT}", *extra]


KEEP = ("step:", "eval_loss", "Train/eval split", "num_learnable_params", "input_features",
        "Traceback", "Error", "error:", "Track this run", "End of training", "out of memory")


def gpu_watch(stop, peak):
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip().splitlines()
        if out:
            peak[0] = max(peak[0], int(out[0]))
        time.sleep(5)


def disk_free_gb(path=None):
    # Falls through so a bad path can never raise inside the watchdog loop, where an exception
    # would silence the very warning this exists to produce.
    for p in ([str(path)] if path else []) + [str(WORK), "/", "."]:
        try:
            s = shutil.disk_usage(p)
            return s.free / 1e9, s.used / 1e9
        except Exception:
            continue
    return float("nan"), float("nan")


def dir_size_gb(path):
    try:
        return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9
    except Exception:
        return 0.0


def watch_disk(stop, out_dir, total_steps, save_freq, period=120):
    """Report disk as the run proceeds and shout BEFORE it runs out.

    Every checkpoint is kept (KEEP_ALL_CHECKPOINTS), so the footprint grows all run: for pi0.5 that
    is ~11.5 GB every save_freq steps. Rather than discover the limit at hour 14, this measures the
    actual size of the checkpoints written so far, extrapolates to the number still to come, and
    compares that against free space.
    """
    d = Path(out_dir) / "checkpoints"
    warned = False
    while not stop.is_set():
        try:
            free, used = disk_free_gb(out_dir)
            done = sorted(int(p.name) for p in d.iterdir()
                          if d.is_dir() and p.is_dir() and not p.is_symlink() and p.name.isdigit()) \
                   if d.is_dir() else []
            if done:
                each = dir_size_gb(d) / len(done)
                remaining = max(0, (total_steps // save_freq) - len(done))
                need = each * remaining
                msg = (f"    [disk] {len(done)} checkpoint(s) written, {each:.1f} GB each | "
                       f"{remaining} to go needs ~{need:.0f} GB | {free:.0f} GB free")
                if need > free * 0.9 and not warned:
                    warned = True
                    print("\n" + "!" * 78, flush=True)
                    print(f"  DISK WARNING: the remaining {remaining} checkpoint(s) need "
                          f"~{need:.0f} GB but only {free:.0f} GB is free.", flush=True)
                    print(f"  This run will stop partway unless you act. Options, cheapest first:", flush=True)
                    print(f"    - grow the data disk (this host charges per GB-day);", flush=True)
                    print(f"    - delete older local checkpoints by hand - they are already on the", flush=True)
                    print(f"      Hub, so nothing is lost:", flush=True)
                    print(f"        import shutil; shutil.rmtree('{d}/<step>')", flush=True)
                    print(f"    - or set KEEP_ALL_CHECKPOINTS = False and restart the cell.", flush=True)
                    print("!" * 78 + "\n", flush=True)
                else:
                    print(msg, flush=True)
        except Exception as e:
            print(f"    [disk] watchdog pass failed ({type(e).__name__}); will retry", flush=True)
        stop.wait(period)


def reap_checkpoints(stop, out_dir, keep=KEEP_CKPTS):
    """Delete superseded LOCAL checkpoints. Only runs when KEEP_ALL_CHECKPOINTS is False.

    Each checkpoint is the full model plus optimizer state - for pi0.5 ~11.5 GB - and lerobot never
    prunes them. They are redundant locally the moment save_checkpoint_to_hub has pushed them, which
    is what makes deleting them safe. Keeps the newest `keep` numbered directories and never touches
    `last`, the symlink resume follows.
    """
    d = Path(out_dir) / "checkpoints"
    while not stop.is_set():
        try:
            if d.is_dir():
                steps = sorted((p for p in d.iterdir() if p.is_dir() and not p.is_symlink()
                                and p.name.isdigit()), key=lambda p: int(p.name))
                dropped = []
                for p in (steps[:-keep] if len(steps) > keep else []):
                    shutil.rmtree(p, ignore_errors=True)   # per-item, so one failure is not fatal
                    dropped.append(p.name)
                if dropped:
                    print(f"    [disk] removed local checkpoint(s) {', '.join(dropped)} "
                          f"(already on the Hub); {disk_free_gb()[0]:.0f} GB free", flush=True)
        except Exception as e:
            print(f"    [disk] reaper pass failed ({type(e).__name__}); will retry", flush=True)
        stop.wait(60)


def run(cmd, quiet=True):
    """Stream a run, dropping tqdm spam, and record peak GPU memory alongside throughput.

    Markers are extracted AS EACH LINE STREAMS BY, not by searching a buffer afterwards. A real run
    emits far more output than it is safe to hold in memory, and everything the smoke test checks -
    the train/eval split, the wandb URL, the parameter count - is printed in the first few seconds,
    so a bounded tail buffer would have evicted all of it long before the run ended.
    """
    t0 = time.time()
    head, tail = [], collections.deque(maxlen=120)   # head for context, tail for failure diagnosis
    peak, stop = [0], threading.Event()
    seen = {"split": None, "wandb": None, "params": None}
    rates, losses, eval_losses = [], [], []
    threading.Thread(target=gpu_watch, args=(stop, peak), daemon=True).start()
    out_dir = next((c.split("=", 1)[1] for c in cmd if c.startswith("--output_dir=")), None)
    n_steps = next((int(c.split("=", 1)[1]) for c in cmd if c.startswith("--steps=")), STEPS)
    if out_dir:
        if KEEP_ALL_CHECKPOINTS:
            threading.Thread(target=watch_disk, args=(stop, out_dir, n_steps, SAVE_FREQ),
                             daemon=True).start()
        else:
            threading.Thread(target=reap_checkpoints, args=(stop, out_dir), daemon=True).start()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    oom = False
    for line in p.stdout:
        if len(head) < 400:
            head.append(line)
        tail.append(line)
        if "out of memory" in line.lower():
            oom = True
        m = re.search(r"Train/eval split: ([0-9]+) train, ([0-9]+) eval", line)
        if m:
            seen["split"] = (int(m.group(1)), int(m.group(2)))
        m = re.search(r"(https://wandb\.ai/[^\s]+)", line)
        if m and not seen["wandb"]:
            seen["wandb"] = m.group(1)
        m = re.search(r"num_learnable_params=([0-9]+)", line)
        if m:
            seen["params"] = int(m.group(1))
        # tqdm FLIPS THE UNIT once an iteration takes over a second: ACT logs "4.47step/s" but
        # pi0.5 logs "1.95s/step". Parse both and normalise to steps/s, or the slow policies
        # report no throughput at all and the readiness verdict fails on a healthy run.
        rates += [float(x) for x in re.findall(r"([0-9.]+) ?step/s", line)]
        rates += [1.0 / float(x) for x in re.findall(r"([0-9.]+) ?s/step", line) if float(x) > 0]
        # " loss:" with the leading space on purpose: it must not also match l1_loss: or kld_loss:
        losses += [float(x) for x in re.findall(r" loss:([0-9.]+)", line)]
        eval_losses += [float(x) for x in re.findall(r"eval_loss=([0-9.]+)", line)]
        if (not quiet) or any(k in line for k in KEEP) or "wandb.ai" in line:
            sys.stdout.write(line); sys.stdout.flush()
    p.wait(); stop.set()
    dt = (time.time() - t0) / 3600
    free, used = disk_free_gb()
    print(f"[exit {p.returncode} after {dt:.2f} h | peak GPU {peak[0]/1024:.1f} GiB "
          f"| disk {used:.0f} GB used, {free:.0f} GB free]")
    if p.returncode != 0:
        print("--- last lines ---"); print("".join(tail)[-2500:])
    return {"code": p.returncode, "hours": dt, "peak_gib": peak[0] / 1024, "oom": oom,
            "rates": rates, "losses": losses, "eval_losses": eval_losses, **seen,
            "head": "".join(head), "tail": "".join(tail)}


print("holdout      :", HOLDOUT)
print("episode order:", EPISODES[:6], "...", EPISODES[-6:])
print("workers      :", NUM_WORKERS)

## 7 — Preflight

Three things that each cost minutes now and hours later: the dataset exists and carries its
codebase-version git tag; the holdout reorder is well formed; and the camera keys line up with what the
policy expects. None of this needs a GPU.

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
checks = []

# ---------------------------------------------------------------------------------------------
# The gated backbone. pi0.5's VLM tokenizer is loaded from a GATED repo whose id is hardcoded in
# lerobot (policies/pi05/processor_pi05.py), and nothing in the pi05 checkpoint bundles it. This
# fails 10 minutes into a run, after the dataset has downloaded and the model is on the GPU, so it
# is worth 3 seconds here. model_info() succeeds even when you lack access - only fetching a file
# actually proves it.
GATED = "google/paligemma-3b-pt-224"
try:
    hf_hub_download(GATED, "config.json")
    gate_ok, gate_why = True, "readable"
except Exception as e:
    gate_ok, gate_why = False, type(e).__name__
checks.append((f"gated backbone {GATED} readable ({gate_why})", gate_ok))
if not gate_ok:
    fg = (whoami().get("auth", {}).get("accessToken", {}).get("fineGrained") or {})
    can_gated = fg.get("canReadGatedRepos")
    print(f"\n{'!' * 78}")
    print(f"  Cannot read {GATED}, which pi0.5 needs for its tokenizer.")
    print(f"  Two separate things are required; yours:")
    print(f"    1. access granted to the repo        -> NO (this is what failed)")
    print(f"    2. token may read gated repos        -> {can_gated}")
    print(f"  Fix 1: open https://huggingface.co/{GATED} and request access.")
    print(f"         The gate is reviewed manually, so it is not instant.")
    if can_gated is False:
        print(f"  Fix 2: edit the token at https://huggingface.co/settings/tokens and tick")
        print(f"         'Read access to contents of all public gated repos you can access'.")
    print(f"  Meanwhile SmolVLA is unblocked - its backbone is not gated.")
    print(f"{'!' * 78}\n")


ok_ds = api.repo_exists(DATASET, repo_type="dataset")
checks.append(("trimmed dataset exists", ok_ds))
assert ok_ds, f"{DATASET} not found. Run colab_phase07_prep_trim.ipynb (Stage 2) first."

info = json.loads(open(hf_hub_download(DATASET, "meta/info.json", repo_type="dataset")).read())
tags = [t.name for t in api.list_repo_refs(DATASET, repo_type="dataset").tags]
# lerobot resolves a dataset at the revision named by its codebase_version; without the tag, training
# dies several minutes in with "Your dataset must be tagged with a codebase version".
checks.append((f"codebase-version tag {info['codebase_version']}", info["codebase_version"] in tags))
checks.append(("50 episodes", info["total_episodes"] == 50))
checks.append(("idle-trimmed (fewer frames than the source's 35,141)", info["total_frames"] < 35141))
print(f"dataset: {info['total_episodes']} episodes, {info['total_frames']} frames, "
      f"{info['fps']} fps, tags={tags}")

checks.append(("holdout reorder is a permutation of 0-49", sorted(EPISODES) == list(range(50))))
checks.append(("holdout is the tail of the episode list", EPISODES[-len(HOLDOUT):] == HOLDOUT))

ds_cams = {k for k in info["features"] if k.startswith("observation.images.")}
# The decisive check for a pretrained policy: after renaming, the batch's camera keys must be a subset
# of what the checkpoint declares. lerobot accepts a subset in either direction, and anything the policy
# declares but the batch does not supply is padded internally.
pol_cfg = json.loads(open(hf_hub_download(BASE_MODEL, "config.json")).read())
pol_cams = {k for k, v in pol_cfg["input_features"].items() if v["type"] == "VISUAL"}
renamed = {RENAME.get(k, k) for k in ds_cams}
print("\npolicy declares :", sorted(pol_cams))
print("dataset supplies:", sorted(ds_cams))
print("after rename    :", sorted(renamed))
print("left unfed      :", sorted(pol_cams - renamed), "(padded internally with a zero mask)")
checks += [
    ("rename map keys exist in the dataset", set(RENAME) <= ds_cams),
    ("renamed keys are a subset of the policy's", renamed <= pol_cams),
]
for k, v in sorted(pol_cfg.items()):
    if k in ("freeze_vision_encoder", "train_expert_only", "optimizer_lr", "chunk_size",
             "n_action_steps", "dtype", "gradient_checkpointing", "resize_imgs_with_padding",
             "image_resolution"):
        print(f"  base config  {k:28s} = {v}")

print()
for n, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n}")
assert all(ok for _, ok in checks), "preflight failed - fix before running Part B"
print("\npreflight OK")

# Phase 0.7 — Stage 3: fine-tune pi0.5 on the portable-rig dataset

One of **three notebooks run in parallel**, one per policy, for the Q2 policy-transfer question of
[08 §A.1](08_data_quality_research.md): *do the same data properties hold across ACT, SmolVLA and π0.5?*
This notebook trains **pi0.5** only.

| | |
|---|---|
| Dataset | `..._20261003_211912_trimmed` — 50 episodes, idle-trimmed by [Stage 2](colab_phase07_prep_trim.ipynb) |
| GPU | **A100 40 GB (required)** |
| Expected runtime | **~6-12 h** — Part B prints the real figure for your GPU |
| Output | `HALDijkstraaa/pi05_tray_cylinder_s1000_trim` (private), checkpointed to the Hub as it goes |
| Siblings | [ACT](colab_phase07_act.ipynb) · [SmolVLA](colab_phase07_smolvla.ipynb) · [π0.5](colab_phase07_pi05.ipynb) |

**π0.5** fine-tunes `lerobot/pi05_base` (3.3 B). The two decisions worth knowing before you start:

- **A full fine-tune is impossible on Colab.** It needs >70 GB (09 §D: 52.8 GB of optimiser states plus
  ~17 GB of activations) and the largest Colab offers is a 40 GB A100. This notebook uses
  **`train_expert_only=true`** — the VLM is frozen and only the action expert and projections train.
- **That choice is deliberate, not just a memory workaround.** It is the same adaptation regime SmolVLA's
  base config uses by default, which keeps the SmolVLA↔π0.5 comparison about model class rather than
  about adaptation method — the confound 09 §D flags for Q2. LoRA is available in Part D as an
  alternative, but it would *introduce* that confound rather than remove it. ACT trains from scratch, so
  the asymmetry against ACT is unavoidable and belongs in the write-up.

It also needs a **rename map**: the checkpoint declares `base_0_rgb` / `left_wrist_0_rgb` /
`right_wrist_0_rgb`. The third is left unfed and π0.5 pads it with a −1 image and a zero attention mask,
which is exactly how openpi serves a one-base-one-wrist arm.

## ⚠ Prerequisite: access to `google/paligemma-3b-pt-224`

**Do this before anything else — it is a waiting game, not a setting.** π0.5's VLM is PaliGemma, and
lerobot loads its *tokenizer* from `google/paligemma-3b-pt-224`, which is a **gated** repo. The id is
hardcoded (`lerobot/policies/pi05/processor_pi05.py:152`) and no π0.5 checkpoint bundles the tokenizer —
not LeRobot's own, not any community SO-100/SO-101 fine-tune — so every working π0.5 fine-tune belongs to
someone who was granted access. There is no workaround worth having: substituting a different tokenizer
would silently change the language conditioning and make this run incomparable to everything else.

1. Open **https://huggingface.co/google/paligemma-3b-pt-224** and request access (accept Google's terms).
   The gate is `manual`, so approval is not instant — request it now and come back.
2. Your token also needs *"Read access to contents of all public gated repos you can access"*
   (`canReadGatedRepos`). Part A checks both and tells you which one is missing.

Until then, run [SmolVLA](colab_phase07_smolvla.ipynb) instead — its backbone
(`HuggingFaceTB/SmolVLM2-500M-Video-Instruct`) is not gated, so it is unblocked.

## Held constant across all three notebooks

Everything that is not the policy, so the comparison means something: the **same trimmed dataset**, the
**same balanced holdout** `[9, 19, 29, 39, 49]`, **seed 1000**, both cameras, no image augmentation,
validation every 5000 steps, and one wandb project (`phase07-policy-transfer`) so the curves overlay.

Per-policy hyperparameters are each policy's own published recipe — that is the intended design
([08 §A.4](08_data_quality_research.md) holds hyperparameters fixed *after* one clean run, not identical
across policies). **Freeze whatever this run lands on and reuse it for the dial grid.**

## How to use

**Part A** once. **Part B** is a ~15 min smoke test that fails loudly rather than warning — paste its
output before starting the long run. **Part C** is one cell: run it and walk away. It is **restartable**,
so after a Colab disconnect you re-run that same cell and it resumes from the last Hub checkpoint.

> **Do not start before Stage 2 prints `READY`.** Every notebook here reads the trimmed dataset; Part A
> checks it exists and is tagged.

In [ ]:
from huggingface_hub import HfApi

SMOKE_REPO, N = f"{HF_USER}/pi05_smoke_delete_me", 300
batch, res = BATCH, None

while True:
    print(f"\n{'=' * 70}\n  smoke test at batch {batch}\n{'=' * 70}")
    shutil.rmtree(OUTPUT_ROOT / "smoke", ignore_errors=True)
    cmd = [c for c in train_cmd(job="smoke", steps=N, batch=batch)
           if not c.startswith(("--policy.repo_id", "--save_freq", "--eval_steps"))]
    cmd += [f"--policy.repo_id={SMOKE_REPO}", f"--eval_steps={N // 2}", f"--save_freq={N}"]
    res = run(cmd)

    if res["oom"] and batch > 1:
        print(f"\n!! OOM at batch {batch}. Halving and retrying.")
        print("   Steps will be scaled to keep samples-seen constant, so the recipe's sample budget")
        print("   is preserved even though the batch is smaller.")
        batch //= 2
        continue
    break

# Every check reports what was actually OBSERVED, so a failure is diagnosable rather than a bare
# FAIL. The values come from run(), which extracts them while the output streams past - searching a
# buffer afterwards is what made the first version report false failures, because the lines it
# looked for scroll away in the first seconds of a run.
split = res["split"]
checks = [
    ("training completed",
     res["code"] == 0),
    (f"balanced split 45 train / 5 eval   (saw {split})",
     split == (45, 5)),
    (f"validation ran   ({len(res['eval_losses'])} eval_loss points: {res['eval_losses']})",
     len(res["eval_losses"]) > 0),
    (f"wandb connected   ({res['wandb'] or 'NO run URL seen'})",
     bool(res["wandb"])),
]
try:
    files = HfApi().list_repo_files(SMOKE_REPO, repo_type="model")
    checks.append(("checkpoint uploaded to the Hub", any("checkpoints/" in f for f in files)))
except Exception as e:
    checks.append((f"checkpoint uploaded to the Hub ({type(e).__name__})", False))

losses, ev = res["losses"], res["eval_losses"]
if len(losses) >= 2:
    checks.append((f"train loss fell ({losses[0]:.3f} -> {losses[-1]:.3f})", losses[-1] < losses[0]))
if len(ev) >= 2:
    checks.append((f"eval loss fell ({ev[0]:.4f} -> {ev[-1]:.4f})", ev[-1] < ev[0]))

print()
for n_, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n_}")

BATCH = batch
# Derived from the published recipe, not from the current STEPS, so re-running this cell after a
# step-down does not scale the step count a second time.
STEPS = int(round(BASE_STEPS * BASE_BATCH / BATCH))
rates = res["rates"]
rate = sorted(rates)[len(rates) // 2] if rates else None

print(f"\n{'=' * 70}")
print(f"  GPU            : {GPU}")
print(f"  peak memory    : {res['peak_gib']:.1f} GiB")
if rate:
    hours = STEPS / rate / 3600
    print(f"  throughput     : {rate:.2f} step/s  ({rate * BATCH:.0f} samples/s)")
    print(f"  batch          : {BATCH}"
          + ("" if BATCH == BASE_BATCH else f"   (stepped down from {BASE_BATCH})"))
    print(f"  steps          : {STEPS:,}   ({STEPS * BATCH / 1000:.0f}k samples)")
    print(f"  PROJECTED RUN  : {hours:.1f} h  (+ validation and checkpoint pushes, roughly 5%)")
    if hours > 12:
        print("\n  !! Over 12 h. Colab will likely disconnect at least once - Part C resumes from")
        print(f"     the last Hub checkpoint, losing at most {SAVE_FREQ:,} steps "
              f"({SAVE_FREQ / rate / 3600:.1f} h), so that is survivable.")
        print("     To shorten it instead, set STEPS by hand (the cosine schedule follows it) or")
        print("     try --policy.gradient_checkpointing=false if peak memory leaves room.")

    # Disk is the other thing that ends a long VLA run. Each checkpoint is the whole model plus
    # optimizer state, and the smoke test has just written exactly one, so measure rather than guess.
    ck = OUTPUT_ROOT / "smoke" / "checkpoints"
    sz = 0
    if ck.is_dir():
        sz = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e9
    free, used = disk_free_gb()
    n_ck = max(1, STEPS // SAVE_FREQ)
    print(f"\n  disk now       : {used:.0f} GB used, {free:.0f} GB free")
    if sz:
        # Everything already on disk stays there (KEEP_ALL_CHECKPOINTS), so what matters is whether
        # the checkpoints still to be written fit in what is free RIGHT NOW.
        need = sz * n_ck
        fits = need < free
        print(f"  per checkpoint : {sz:.1f} GB  x {n_ck} checkpoints over the run = {need:.0f} GB")
        print(f"  keeping all    : {'FITS' if fits else 'WILL NOT FIT'}  "
              f"({need:.0f} GB needed vs {free:.0f} GB free, "
              f"{free - need:+.0f} GB margin at the end)")
        if not fits:
            print(f"\n  !! The run would stop partway. Either grow the data disk by "
                  f"~{need - free + 10:.0f} GB, raise SAVE_FREQ to write fewer checkpoints, or set")
            print(f"     KEEP_ALL_CHECKPOINTS = False to retain only the newest {KEEP_CKPTS}.")
        elif free - need < 2 * sz:
            print(f"     Margin is under two checkpoints - the watchdog in Part C will warn you if")
            print(f"     it tightens. Deleting an early checkpoint by hand is always safe; they are")
            print(f"     all on the Hub.")
        print(f"  Hub repo ends  : ~{need:.0f} GB, since every checkpoint is pushed (Part D prunes).")
else:
    print("  throughput     : could not parse step/s from the log")

SMOKE_OK = all(ok for _, ok in checks) and rate is not None
print(f"\n  READY: {SMOKE_OK}")
print("=" * 70)
if SMOKE_OK:
    # Reclaim the smoke run's own checkpoint before Part C starts. For pi0.5 that is ~11.5 GB,
    # a meaningful slice of the margin when every checkpoint is being kept.
    _before = disk_free_gb()[0]
    shutil.rmtree(OUTPUT_ROOT / "smoke", ignore_errors=True)
    _after = disk_free_gb()[0]
    print(f"\nsmoke output cleaned up: {_after - _before:+.1f} GB reclaimed, {_after:.0f} GB free")
    print("\nPaste the block above, then run Part C.")
    print(f"Clean up when convenient:  HfApi().delete_repo('{SMOKE_REPO}', repo_type='model')")
else:
    raise RuntimeError("smoke test failed - see the FAIL lines above. Do not start Part C.")

# Phase 0.7 — Stage 3: fine-tune pi0.5 on the portable-rig dataset

One of **three notebooks run in parallel**, one per policy, for the Q2 policy-transfer question of
[08 §A.1](08_data_quality_research.md): *do the same data properties hold across ACT, SmolVLA and π0.5?*
This notebook trains **pi0.5** only.

| | |
|---|---|
| Dataset | `..._20261003_211912_trimmed` — 50 episodes, idle-trimmed by [Stage 2](colab_phase07_prep_trim.ipynb) |
| GPU | **A100 40 GB (required)** |
| Expected runtime | **~6-12 h** — Part B prints the real figure for your GPU |
| Output | `HALDijkstraaa/pi05_tray_cylinder_s1000_trim` (private), checkpointed to the Hub as it goes |
| Siblings | [ACT](colab_phase07_act.ipynb) · [SmolVLA](colab_phase07_smolvla.ipynb) · [π0.5](colab_phase07_pi05.ipynb) |

**π0.5** fine-tunes `lerobot/pi05_base` (3.3 B). The two decisions worth knowing before you start:

- **A full fine-tune is impossible on Colab.** It needs >70 GB (09 §D: 52.8 GB of optimiser states plus
  ~17 GB of activations) and the largest Colab offers is a 40 GB A100. This notebook uses
  **`train_expert_only=true`** — the VLM is frozen and only the action expert and projections train.
- **That choice is deliberate, not just a memory workaround.** It is the same adaptation regime SmolVLA's
  base config uses by default, which keeps the SmolVLA↔π0.5 comparison about model class rather than
  about adaptation method — the confound 09 §D flags for Q2. LoRA is available in Part D as an
  alternative, but it would *introduce* that confound rather than remove it. ACT trains from scratch, so
  the asymmetry against ACT is unavoidable and belongs in the write-up.

It also needs a **rename map**: the checkpoint declares `base_0_rgb` / `left_wrist_0_rgb` /
`right_wrist_0_rgb`. The third is left unfed and π0.5 pads it with a −1 image and a zero attention mask,
which is exactly how openpi serves a one-base-one-wrist arm.

## ⚠ Prerequisite: access to `google/paligemma-3b-pt-224`

**Do this before anything else — it is a waiting game, not a setting.** π0.5's VLM is PaliGemma, and
lerobot loads its *tokenizer* from `google/paligemma-3b-pt-224`, which is a **gated** repo. The id is
hardcoded (`lerobot/policies/pi05/processor_pi05.py:152`) and no π0.5 checkpoint bundles the tokenizer —
not LeRobot's own, not any community SO-100/SO-101 fine-tune — so every working π0.5 fine-tune belongs to
someone who was granted access. There is no workaround worth having: substituting a different tokenizer
would silently change the language conditioning and make this run incomparable to everything else.

1. Open **https://huggingface.co/google/paligemma-3b-pt-224** and request access (accept Google's terms).
   The gate is `manual`, so approval is not instant — request it now and come back.
2. Your token also needs *"Read access to contents of all public gated repos you can access"*
   (`canReadGatedRepos`). Part A checks both and tells you which one is missing.

Until then, run [SmolVLA](colab_phase07_smolvla.ipynb) instead — its backbone
(`HuggingFaceTB/SmolVLM2-500M-Video-Instruct`) is not gated, so it is unblocked.

## Held constant across all three notebooks

Everything that is not the policy, so the comparison means something: the **same trimmed dataset**, the
**same balanced holdout** `[9, 19, 29, 39, 49]`, **seed 1000**, both cameras, no image augmentation,
validation every 5000 steps, and one wandb project (`phase07-policy-transfer`) so the curves overlay.

Per-policy hyperparameters are each policy's own published recipe — that is the intended design
([08 §A.4](08_data_quality_research.md) holds hyperparameters fixed *after* one clean run, not identical
across policies). **Freeze whatever this run lands on and reuse it for the dial grid.**

## How to use

**Part A** once. **Part B** is a ~15 min smoke test that fails loudly rather than warning — paste its
output before starting the long run. **Part C** is one cell: run it and walk away. It is **restartable**,
so after a Colab disconnect you re-run that same cell and it resumes from the last Hub checkpoint.

> **Do not start before Stage 2 prints `READY`.** Every notebook here reads the trimmed dataset; Part A
> checks it exists and is tagged.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

_api = HfApi()

if not globals().get("SMOKE_OK"):
    print("!! Part B has not run in this kernel (SMOKE_OK is unset).")
    print("   Expected after a reconnect: this cell is restartable and re-checks the Hub itself.")
    print("   On a fresh setup, run Part B first - it must print READY: True.\n")


def run_state(job):
    """Where this run stands, read from its Hub repo: finished, part-done, or not started."""
    try:
        files = _api.list_repo_files(f"{HF_USER}/{job}", repo_type="model")
    except Exception:
        return "new", None
    if "model.safetensors" in files:            # the final push only happens at the very end
        return "done", None
    steps = sorted({f.split("/")[1] for f in files
                    if f.startswith("checkpoints/") and f.split("/")[1].isdigit()})
    return ("resume", steps[-1]) if steps else ("new", None)

t0 = time.time()
where, last = run_state(JOB)

if where == "done":
    print(f"already finished on the Hub - nothing to do: https://huggingface.co/{HF_USER}/{JOB}")
    res = {"code": 0, "hours": 0.0}
elif where == "resume":
    # Prefer the local checkpoint if this VM still has it; otherwise pull from the Hub. The step
    # counter, optimizer state, episode order, holdout and seed all come back from the checkpoint -
    # only --steps is read from the CLI.
    local = OUTPUT_ROOT / JOB / "checkpoints/last/pretrained_model/train_config.json"
    src = str(local) if local.exists() else f"{HF_USER}/{JOB}"
    print(f"resuming from step {last} ({'local' if local.exists() else 'Hub'})")
    cmd = ["lerobot-train", f"--config_path={src}", "--resume=true", f"--steps={STEPS}"]
    if not local.exists():
        cmd.append(f"--output_dir={OUTPUT_ROOT}/{JOB}_resumed")
    res = run(cmd)
else:
    print(f"starting fresh: {STEPS:,} steps at batch {BATCH}")
    res = run(train_cmd(steps=STEPS, batch=BATCH))

status = "ok" if res["code"] == 0 else f"FAILED (exit {res['code']})"
print(f"\n{'=' * 70}")
print(f"  {JOB}")
print(f"  status : {status}")
print(f"  time   : {res['hours']:.2f} h")
print(f"  model  : https://huggingface.co/{HF_USER}/{JOB}")
print(f"  wandb  : project {PROJECT}, run {JOB}")
print("=" * 70)
if res["code"] != 0:
    print("\nRe-run this cell to resume from the last Hub checkpoint, or see Part D.")
else:
    print("\nPaste this summary back. Then run the other two policies' notebooks if they are not")
    print("already going, and record the result in Details/phase07_training.md.")

# Phase 0.7 — Stage 3: fine-tune pi0.5 on the portable-rig dataset

One of **three notebooks run in parallel**, one per policy, for the Q2 policy-transfer question of
[08 §A.1](08_data_quality_research.md): *do the same data properties hold across ACT, SmolVLA and π0.5?*
This notebook trains **pi0.5** only.

| | |
|---|---|
| Dataset | `..._20261003_211912_trimmed` — 50 episodes, idle-trimmed by [Stage 2](colab_phase07_prep_trim.ipynb) |
| GPU | **A100 40 GB (required)** |
| Expected runtime | **~6-12 h** — Part B prints the real figure for your GPU |
| Output | `HALDijkstraaa/pi05_tray_cylinder_s1000_trim` (private), checkpointed to the Hub as it goes |
| Siblings | [ACT](colab_phase07_act.ipynb) · [SmolVLA](colab_phase07_smolvla.ipynb) · [π0.5](colab_phase07_pi05.ipynb) |

**π0.5** fine-tunes `lerobot/pi05_base` (3.3 B). The two decisions worth knowing before you start:

- **A full fine-tune is impossible on Colab.** It needs >70 GB (09 §D: 52.8 GB of optimiser states plus
  ~17 GB of activations) and the largest Colab offers is a 40 GB A100. This notebook uses
  **`train_expert_only=true`** — the VLM is frozen and only the action expert and projections train.
- **That choice is deliberate, not just a memory workaround.** It is the same adaptation regime SmolVLA's
  base config uses by default, which keeps the SmolVLA↔π0.5 comparison about model class rather than
  about adaptation method — the confound 09 §D flags for Q2. LoRA is available in Part D as an
  alternative, but it would *introduce* that confound rather than remove it. ACT trains from scratch, so
  the asymmetry against ACT is unavoidable and belongs in the write-up.

It also needs a **rename map**: the checkpoint declares `base_0_rgb` / `left_wrist_0_rgb` /
`right_wrist_0_rgb`. The third is left unfed and π0.5 pads it with a −1 image and a zero attention mask,
which is exactly how openpi serves a one-base-one-wrist arm.

## ⚠ Prerequisite: access to `google/paligemma-3b-pt-224`

**Do this before anything else — it is a waiting game, not a setting.** π0.5's VLM is PaliGemma, and
lerobot loads its *tokenizer* from `google/paligemma-3b-pt-224`, which is a **gated** repo. The id is
hardcoded (`lerobot/policies/pi05/processor_pi05.py:152`) and no π0.5 checkpoint bundles the tokenizer —
not LeRobot's own, not any community SO-100/SO-101 fine-tune — so every working π0.5 fine-tune belongs to
someone who was granted access. There is no workaround worth having: substituting a different tokenizer
would silently change the language conditioning and make this run incomparable to everything else.

1. Open **https://huggingface.co/google/paligemma-3b-pt-224** and request access (accept Google's terms).
   The gate is `manual`, so approval is not instant — request it now and come back.
2. Your token also needs *"Read access to contents of all public gated repos you can access"*
   (`canReadGatedRepos`). Part A checks both and tells you which one is missing.

Until then, run [SmolVLA](colab_phase07_smolvla.ipynb) instead — its backbone
(`HuggingFaceTB/SmolVLM2-500M-Video-Instruct`) is not gated, so it is unblocked.

## Held constant across all three notebooks

Everything that is not the policy, so the comparison means something: the **same trimmed dataset**, the
**same balanced holdout** `[9, 19, 29, 39, 49]`, **seed 1000**, both cameras, no image augmentation,
validation every 5000 steps, and one wandb project (`phase07-policy-transfer`) so the curves overlay.

Per-policy hyperparameters are each policy's own published recipe — that is the intended design
([08 §A.4](08_data_quality_research.md) holds hyperparameters fixed *after* one clean run, not identical
across policies). **Freeze whatever this run lands on and reuse it for the dial grid.**

## How to use

**Part A** once. **Part B** is a ~15 min smoke test that fails loudly rather than warning — paste its
output before starting the long run. **Part C** is one cell: run it and walk away. It is **restartable**,
so after a Colab disconnect you re-run that same cell and it resumes from the last Hub checkpoint.

> **Do not start before Stage 2 prints `READY`.** Every notebook here reads the trimmed dataset; Part A
> checks it exists and is tagged.